In [175]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb
# pypdf is for simple pdf while pymupdf is for complex pdfs
# sentence-transformers is for embeddings 
# chromadb is a vector data base

In [176]:
from langchain_core.documents import Document


In [177]:
# getting the text data
from langchain_community.document_loaders.text import TextLoader
#loader = TextLoader("data/python.txt", encoding="utf-8")
#document = loader.load()

In [178]:
# getting the pdf
from langchain_community.document_loaders.pdf import PyPDFLoader
#loader = PyPDFLoader("data/research2.pdf")
#document= loader.load() 


# Ingestion Pipeline

In [179]:
# to convert the Data into documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

def load_all_pdfs():
    folder_path = "Data/"
    num_docs = 0
    all_docs =[]
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file_path
            pdf_path= os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs +=1

    print("total Pdfs", num_docs)
    print("total Pages", len(all_docs))
    return all_docs

In [180]:
all_documents = load_all_pdfs()


total Pdfs 2
total Pages 32


# Convert the document into chunks

In [181]:
# !pip install langchain_text_splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_docs(documents, chunk_size = 500, chunk_overlap = 50):
    text_spliter = RecursiveCharacterTextSplitter(
        chunk_overlap= chunk_overlap,
        chunk_size=chunk_size
    )
    splited_text = text_spliter.split_documents(documents)
    return splited_text

In [182]:
chunks = split_docs(all_documents)


# Embeddings

In [183]:
from sentence_transformers import SentenceTransformer

In [184]:
class EmbeddingManager:
    def __init__(self, model_name= "all-MiniLM-L6-v2"):

        self.model_name = model_name
        print("loading_model...", self.model_name)
        self.model= SentenceTransformer(self.model_name) # we used sentencetransformer to load the model
        print("embeddings Dimmentions", self.model.get_embedding_dimension())
        
    def generate_embeddings(self,text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("empedding Shape", embeddings.shape)
        return embeddings
    

In [185]:
embedding_manager = EmbeddingManager()

loading_model... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embeddings Dimmentions 384


# Vector Store Manager

In [186]:
import chromadb
import uuid # for unique indexing

In [187]:
class vectorstoremanager:
    def __init__(self, collection_name= "pdf_documents", persist_directory = "Data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self.initialize_store()
    def initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok= True)

        # create a client

        self.client = chromadb.PersistentClient(path=self.persist_directory)
        
        # create a collection 

        self.collection = self.client.get_or_create_collection(
            name= self.collection_name,
            metadata= {"discription" : "vector store collection for pdf embeddings in RAG"}
        )

        print("iniialize the vector store with collection", self.collection_name)
        print("docs in collection are ", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents are not equall to the number of embeddings")

        ##>> we need to store the new ids, embeddings, document, metadata
        ids = []
        all_metadata = []
        documents_content= []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["docs_index"] = i
            metadata ["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        self.collection.add(
                ids = ids, 
                metadatas = all_metadata,
                documents = documents_content,
                embeddings= embeddings_list
            )
        print("Total documents added in collection", len(documents_content))
        print("docs in collection", self.collection.count())

In [188]:
vector_store = vectorstoremanager()

iniialize the vector store with collection pdf_documents
docs in collection are  1600


In [189]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks, embeddings)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

empedding Shape (320, 384)
Total documents added in collection 320
docs in collection 1920


# Retrieval Pipeline

In [190]:
from sklearn.metrics.pairwise import cosine_similarity


In [191]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager= embedding_manager
        self.vector_store= vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # first convert the query to embeddings
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0] # because it expects a list 

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        # cosine similarity
        retrieved_docs = []
        if (results["documents"] and results["documents"][0]):

            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1-distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append ( {
                        "id": doc_id,
                        "document" : document,
                        "metadata" : metadata,
                        "distance" : distance,
                        "similarity_score": similarity_score,
                        "rank" : i+1
                    })
            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("No documents found")
        return retrieved_docs
                                                                      

In [192]:
RagRetriever = RAGRetriever(embedding_manager, vector_store)
RagRetriever.retrieve("what is attention")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

empedding Shape (1, 384)
retrieved 5 documents


[{'id': 'doc_7552a0fc-5453-4cf3-a570-92292caa75ff',
  'document': 'sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This\nmasking, combined with fact that the output embeddings are offset by one position, ensures that the\npredictions for positioni can depend only on the known outputs at positions less thani.\n3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum',
  'metadata': {'source': 'Data/research1.pdf',
   'lastpage': '6008',
   'content_length': 499,
   'firstpage': '5998',
   'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett',
   'docs_index': 263,
   'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin',
   'description-abstract':

# OpenAI-GPT


In [193]:
API_KEY_OPENAI= ""
#!pip install langchain-openai

In [194]:
from langchain_openai import ChatOpenAI

In [198]:
llm = ChatOpenAI(
    openai_api_key = API_KEY_OPENAI,
    model = "gpt-5.4-mini",
    temperature = 0.1, # this defines how creative our model should be
    max_tokens = 1024
)

In [206]:
# Generate our Retrieval augmented ouput
def generate_output(query, RagRetriever, llm, top_k=3):
    results = RagRetriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results ])if results else ""

    if not context:
        print("we found no relevent context")
    # context + query

    prompt = f""" use given context to generate the answer fof the query
            context : {context}
            query : {query}"""
    response = llm.invoke(prompt) # expecting a string as prompt
    return response

In [207]:
answer = generate_output("what is attention", RagRetriever, llm)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

empedding Shape (1, 384)
retrieved 3 documents


AttributeError: 'AIMessage' object has no attribute 'output_text'

# With Groq

In [203]:
print(answer)

content='Attention is a mechanism that maps a **query** and a set of **key-value pairs** to an **output** vector. The output is computed as a **weighted sum of the values**, where the weights are determined by how well the query matches the keys.\n\nIn simple terms, attention lets a model focus on the most relevant parts of the input when producing an output.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 338, 'total_tokens': 416, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DpEx6m3lPcGiGFaklC9vkLEvxx7Ym', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019eb21b-04ec-7613-aa7f-94e24a71bdd8-0' tool_calls=[] invalid_too

In [ ]:
#!pip install langchain-groq

In [205]:
from langchain_groq inport ChatGroq
llm = ChatOpenAI(
    groq_api_key = API_KEY_GROQ,
    model = "qwen/qwen3-32b",
    temperature = 0.1, # this defines how creative our model should be
    max_tokens = 1024
)
# Generate our Retrieval augmented ouput
def generate_output(query, RagRetriever, llm, top_k=3):
    results = RagRetriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results ])if results else ""

    if not context:
        print("we found no relevent context")
    # context + query

    prompt = f""" use given context to generate the answer fof the query
            context : {context}
            query : {query}"""
    response = llm.invoke(prompt.format(context=context, query=query) # expecting a string as prompt
    return response.output_text

SyntaxError: invalid syntax (1331636304.py, line 1)

In [204]:
answer = generate_output("what is RAG", RagRetriever, llm)
print(answer.output_text)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

empedding Shape (1, 384)
retrieved 3 documents


AttributeError: 'AIMessage' object has no attribute 'output_text'